<a href="https://www.kaggle.com/code/chiro999/transformerpreset?scriptVersionId=309287660" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# import OS and check if dataset is mounted
import os

print("Datasets mounted at /kaggle/input:")
print(os.listdir("/kaggle/input"))
print(os.listdir("/kaggle/input/datasets")[:50])
print(os.listdir("/kaggle/input/datasets/organizations")[:100])

Datasets mounted at /kaggle/input:
['datasets']
['organizations']
['nih-chest-xrays']


In [2]:
# Storing root path of ChestXray14 dataset
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays"
print(os.listdir(DATA_ROOT)[:200])

['data']


In [3]:
# Storing specific root path of ChestXray14 dataset and verifying the existence of image files
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

first_png = None
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.endswith(".png"):
            first_png = os.path.join(root, f)
            break
    if first_png:
        break

print("First PNG found:", first_png)


First PNG found: /kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images/00006199_010.png


In [4]:
# Checking whether a CUDA capable GPU is available
!nvidia-smi

Mon Apr  6 02:19:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# importing timm for Vision Transformer 
!pip -q install timm

# Importing PIL to open images and tqdm for progress bars as well as random for reproducibilty
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# PyTorch and its neural network tools.
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# timm for loading the ViT model
import timm
from torchvision import transforms
from sklearn.metrics import roc_auc_score

In [6]:
# Pointing to the metadata csv that contains image names, patient IDs, and label
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

# The 14 chest disease classes
LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]

# Creating a dictionary for these labels
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

# use a same seed to reproduce results
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# Select GPU if available otherwise GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [7]:
# collect all subdirectories inside DATA_ROOT that match 'images_' and contain an 'images' folder
img_dirs = [
    os.path.join(DATA_ROOT, d, "images")
    for d in os.listdir(DATA_ROOT)
    if d.startswith("images_") and os.path.isdir(os.path.join(DATA_ROOT, d, "images"))
]

# print number of image directories found and preview a few paths
print("Image folders found:", len(img_dirs))
print("Example dirs:", img_dirs[:3])

# create a dictionary mapping image filenames to their full file paths
name2path = {}
for d in img_dirs:
    for f in os.listdir(d):
        if f.endswith(".png"):
            name2path[f] = os.path.join(d, f)


# print total number of indexed images and check if a sample file exists
print("Indexed images:", len(name2path))
print("Has 00000001_000.png?", "00000001_000.png" in name2path)

Image folders found: 12
Example dirs: ['/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_012/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_009/images']
Indexed images: 112120
Has 00000001_000.png? True


In [8]:
# load dataset from CSV file into a DataFrame
df = pd.read_csv(CSV_PATH)

# get unique patient IDs and shuffle them for random splitting
patients = df["Patient ID"].unique()
np.random.shuffle(patients)

# split sizes (80% train, 10% validation, 10% test)
n = len(patients)
train_ids = set(patients[: int(0.8*n)])
val_ids   = set(patients[int(0.8*n): int(0.9*n)])
test_ids  = set(patients[int(0.9*n):])

# split the DataFrame based on patient IDs to avoid data leakage
train_df = df[df["Patient ID"].isin(train_ids)].copy()
val_df   = df[df["Patient ID"].isin(val_ids)].copy()
test_df  = df[df["Patient ID"].isin(test_ids)].copy()

# print number of samples in each split
print(len(train_df), len(val_df), len(test_df))

89703 11221 11196


In [9]:
class ChestXray14(Dataset):
    # store dataframe and reset index for consistent access
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    # return total number of samples
    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        # get row corresponding to index i
        row = self.df.iloc[i]

        # extract image filename and retrieve full path
        fname = row["Image Index"]
        img_path = name2path[fname]
        
         # load image in grayscale and convert to 3-channel RGB
        img = Image.open(img_path).convert("L")
        img = Image.merge("RGB", (img, img, img))

        # initialize multi-label target vector
        y = np.zeros(NUM_LABELS, dtype=np.float32)

        # set labels to 1 for each disease present
        for f in str(row["Finding Labels"]).split("|"):
            if f in label2idx:
                y[label2idx[f]] = 1.0
                
        # apply data transformations if provided
        if self.transform:
            img = self.transform(img)
            
        # return image tensor and corresponding label tensor
        return img, torch.tensor(y)

IMG_SIZE = 224
# training data transformations (data augmentation + normalization)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

# validation data transformations (no augmentation, only resizing and normalization)
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

In [10]:
# define batch size and number of parallel workers for data loading
BATCH_SIZE = 32
NUM_WORKERS = 2

# create DataLoader for training set with shuffling for randomness
train_loader = DataLoader(ChestXray14(train_df, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

# create DataLoader for validation set (no shuffling for consistent evaluation)
val_loader = DataLoader(ChestXray14(val_df, val_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# create DataLoader for test set (no shuffling, same as validation) 
test_loader = DataLoader(ChestXray14(test_df, val_tf),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

In [11]:
def compute_pos_weight(df_):
     # initialize count of positive samples for each label
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    # count occurrences of each label across the dataset
    for labels in df_["Finding Labels"].astype(str).values:
        for f in labels.split("|"):
            if f in label2idx:
                counts[label2idx[f]] += 1
    # total number of samples
    N = len(df_)
    # avoid division by zero by clipping minimum positive count to 1
    pos = np.clip(counts, 1.0, None)
    # compute number of negative samples per label
    neg = N - counts
    return torch.tensor(neg / pos, dtype=torch.float32)

# return class weights (negative/positive) for handling class imbalance
pos_weight = compute_pos_weight(train_df).to(device)
print("pos_weight:", pos_weight)

pos_weight: tensor([  8.7609,  39.6448,   7.5310,   4.5925,  18.7889,  16.7630,  76.3969,
         19.5694,  23.1332,  48.3959,  42.3767,  63.4418,  32.6344, 466.2031],
       device='cuda:0')


In [12]:
# define Vision Transformer model
MODEL_NAME = "vit_base_patch16_224"  
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_LABELS).to(device)

# define loss function with class imbalance weighting
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# define optimizer (AdamW with weight decay for regularization)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.05)

# initialize gradient scaler for mixed precision training 
scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda"))

def train_one_epoch():
    # set model to training mode
    model.train()
    total = 0.0
    # iterate over training batches
    for x, y in tqdm(train_loader, leave=False):
        # move data to device (CPU/GPU)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        # reset gradients
        optimizer.zero_grad(set_to_none=True)
        # forward pass with automatic mixed precision
        with torch.amp.autocast("cuda", enabled=(device=="cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        # backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        # accumulate total loss
        total += loss.item() * x.size(0)
    # return average loss over dataset
    return total / len(train_loader.dataset)

@torch.no_grad()
def eval_auc_per_class(model, loader, device, label_names):
    """Returns (mean_auc, per_class_aucs_list) aligned with label_names."""
    # set model to evaluation mode
    model.eval()
    all_logits, all_y = [], []

    # iterate over validation/test batches
    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        # forward pass and move outputs to CPU
        logits = model(x).float().cpu()
        all_logits.append(logits)
        all_y.append(y.cpu())

    # concatenate all predictions and labels
    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy()
    # apply sigmoid to convert logits to probabilities
    y_prob = 1 / (1 + np.exp(-logits))  # sigmoid

    # compute AUC for each class
    per_class = []
    for i in range(len(label_names)):
        if len(np.unique(y_true[:, i])) < 2:
            per_class.append(np.nan)
        else:
            per_class.append(roc_auc_score(y_true[:, i], y_prob[:, i]))
    # compute mean AUC across all classes (ignoring NaNs)
    mean_auc = float(np.nanmean(per_class))
    return mean_auc, per_class

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [13]:
# define number of training epochs and initialize best validation score
EPOCHS = 5
best_val = -1

# path to save the best model checkpoint
best_path = "/kaggle/working/best_custom_vit_chestxray14.pt"

# training loop over epochs
for epoch in range(1, EPOCHS+1):
    # train model for one epoch and compute training loss
    tr_loss = train_one_epoch()
     # evaluate model on validation set using mean AUROC
    val_mean_auc, _ = eval_auc_per_class(model, val_loader, device, LABELS)

    # print training progress
    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} | val mean AUROC {val_mean_auc:.4f}")

    # save model if validation performance improves
    if val_mean_auc > best_val:
        best_val = val_mean_auc
        torch.save(model.state_dict(), best_path)
        print("saved best:", best_path)
        
# print best validation AUROC achieved
print("Best val mean AUROC:", best_val)

Epoch 1/5 | train loss 1.1682 | val mean AUROC 0.7842
saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 2/5 | train loss 1.0499 | val mean AUROC 0.7994
saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 3/5 | train loss 0.9984 | val mean AUROC 0.8126
saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 4/5 | train loss 0.9704 | val mean AUROC 0.8156
saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 5/5 | train loss 0.9311 | val mean AUROC 0.8123
Best val mean AUROC: 0.815602859024118


In [14]:
# load the best saved model weights
model.load_state_dict(torch.load(best_path, map_location=device))

# evaluate model performance on the test set using mean AUROC and per-class AUROC
test_mean_auc, test_aucs = eval_auc_per_class(model, test_loader, device, LABELS)

# print overall test performance
print("Test mean AUROC:", test_mean_auc)

# print AUROC for each individual class
for name, auc in zip(LABELS, test_aucs):
    print(f"{name:18s}: {auc:.4f}")


Test mean AUROC: 0.8203027769841905
Atelectasis       : 0.7904
Cardiomegaly      : 0.9055
Effusion          : 0.8720
Infiltration      : 0.7047
Mass              : 0.8349
Nodule            : 0.7454
Pneumonia         : 0.7790
Pneumothorax      : 0.8468
Consolidation     : 0.7998
Edema             : 0.8844
Emphysema         : 0.8958
Fibrosis          : 0.8015
Pleural_Thickening: 0.7930
Hernia            : 0.8312
